<link rel="stylesheet" href="notebooks/styles.css">

<div class="title-wrap">
  <h1 class="title-main" style="font-weight: bold; font-size: 2.65rem; margin-bottom: 0.5rem;">
  Spatial Data Science Approaches to Wildfire Severity Modeling
</h1>
<h2 class="title-sub" style="font-style: italic; font-size: 1.8rem; margin-top: 0rem; margin-bottom: 0.2rem;">
  A GIS‑Driven, Tree‑Based Machine Learning Analysis of California Wildfires
</h2>
</div>

# Module 7A: *Fire Ignition Class Balancing*
##### Version Number: 4.0
---
### Contents  
> 1. Data Preparation
> 2. Automatic Class Balancing 
> 3. Export File
---
### Notes

**WARNING** this module is computation heavy
- Start with a **baseline model** for comparison.
- Test with multi-classification **tree-based models** (Random Forest, XGBoost) and LGBM.
- Test and evaluate multiple class balancing strategies (No sampling, Oversampling, Undersampling)
- Compare average F1 score among all classes

---
### Inputs
- `X_scaled`,`y_reduced` Reduced or full size Modeling sets

---
### Outputs  

`best_strategy` - dataframe recording best sampling strategy for each method

---
### User Created Dependencies

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

from src.data_utils import *
from src.model_utils import *
from src.plot_utils import *

---
### Third Party Dependencies

In [2]:
# Core Python libraries
import numpy as np
import pandas as pd
import json
# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing and modeling
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

# Style
sns.set(style='whitegrid')
plt.rcParams["figure.dpi"] = 100

---

In [3]:
X_ignition = pd.read_csv('../data/processed/X_ignition.csv')
y_ignition = pd.read_csv('../data/processed/y_ignition.csv').squeeze()

with open('model_parameters_ignition.json', 'r') as f:
    model_parameters_ignition = json.load(f)

## 1. Prepare Data for Modeling - Scaling, Splitting, and Resampling

In [4]:
# Build Final tuned models
ignition_rf = RandomForestClassifier(**model_parameters_ignition['Random Forest'])
ignition_xgb = xgb.XGBClassifier(**model_parameters_ignition['XGBoost'])

In [5]:
models = {
    'Ignition RF' : ignition_rf,
    'Ignition XGB' : ignition_xgb,
}

---

## 2. Automatic Class Balancing

To address the extreme inbalance in the dataset, multiple strategies are explored.
- **In Method Balancing** is used when applicable 
- **RandomUnderSampler** will remove random members of the majority class (Low severity) until they are balanced. This creates a much smaller dataset to model.
- **SMOTE (Synthetic Minority Over-sampling Technique)** will generate synthetic members of the minority classes. Introduces potential noise by synthetic sampling

In [6]:
reform = pd.concat([X_ignition,y_ignition], axis=1)
subset = balancing_subset_df(reform, 'Target_Ignition', 10000)

y = subset['Target_Ignition']
X = subset.drop(columns='Target_Ignition')

9000
700
300


In [7]:
y.value_counts()

Target_Ignition
0    9000
1     700
2     300
Name: count, dtype: int64

In [8]:
# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

In [9]:
sampling_strategies = ['Undersampling','No_balance','Oversampling']

all_results = []
counter = 1

for strategy in sampling_strategies:
    for name, model in models.items():
        print("running", strategy, counter, "of 6...")
        counter = counter + 1
        result = class_balancing(
            X_train, y_train, X_test, y_test,
            model,
            sampling_strategy=strategy
        )
        result['Model_Label'] = name
        all_results.append(result)

# Combine into one DataFrame
all_results_df = pd.concat(all_results, axis=0).reset_index(drop=True)

running Undersampling 1 of 6...


running Undersampling 2 of 6...


running No_balance 3 of 6...


running No_balance 4 of 6...


running Oversampling 5 of 6...


running Oversampling 6 of 6...


In [10]:
all_results_df

,Class,Category,Precision,Recall,F1-Score,Phase,Model,Balancing,Model_Label
0,0,0,0.708473,0.709729,0.706345,Train,RandomForestClassifier,Undersampling,Ignition RF
1,1,1,0.484168,0.493886,0.485310,Train,RandomForestClassifier,Undersampling,Ignition RF
2,2,2,0.622279,0.591476,0.605456,Train,RandomForestClassifier,Undersampling,Ignition RF
3,0,0,0.977183,0.730170,0.835808,Test,RandomForestClassifier,Undersampling,Ignition RF
4,1,1,0.148571,0.495238,0.228571,Test,RandomForestClassifier,Undersampling,Ignition RF
5,2,2,0.225352,0.695652,0.340426,Test,RandomForestClassifier,Undersampling,Ignition RF
6,0,0,0.706923,0.657210,0.677852,Train,XGBClassifier,Undersampling,Ignition XGB
7,1,1,0.453563,0.459297,0.452355,Train,XGBClassifier,Undersampling,Ignition XGB
8,2,2,0.586785,0.613621,0.599207,Train,XGBClassifier,Undersampling,Ignition XGB
9,0,0,0.975806,0.717569,0.826997,Test,XGBClassifier,Undersampling,Ignition XGB


### 2.4 Format and display results

In [11]:
strategy_summary = (
    all_results_df[all_results_df['Phase'] == 'Test']
    .groupby(['Balancing','Model_Label'])['F1-Score']
    .mean()
    .reset_index()
    .sort_values(by='F1-Score', ascending=False)
)

strategy_summary_pivot = (
    strategy_summary
    .pivot(index='Model_Label', columns='Balancing', values='F1-Score')
    .sort_values(by='No_balance', ascending=False)
)

strategy_summary_pivot.style.highlight_max(axis=1, color='lightgreen')

Balancing,No_balance,Oversampling,Undersampling
Model_Label,,,
Ignition RF,0.527409,0.535575,0.468268
Ignition XGB,0.469796,0.513565,0.440514


### Display Best Strategies

In [12]:
# Find best balancing strategy per model
best_strategy = strategy_summary_pivot.idxmax(axis=1)

# Combine with model names into a new DataFrame
best_strategy_df = pd.DataFrame({
    'Model_Label': best_strategy.index,
    'Best_Strategy': best_strategy.values
})

In [13]:
best_strategy_df

,Model_Label,Best_Strategy
0,Ignition RF,Oversampling
1,Ignition XGB,Oversampling


### Detailed results (ALL RESULTS)

In [14]:
all_results_df

,Class,Category,Precision,Recall,F1-Score,Phase,Model,Balancing,Model_Label
0,0,0,0.708473,0.709729,0.706345,Train,RandomForestClassifier,Undersampling,Ignition RF
1,1,1,0.484168,0.493886,0.485310,Train,RandomForestClassifier,Undersampling,Ignition RF
2,2,2,0.622279,0.591476,0.605456,Train,RandomForestClassifier,Undersampling,Ignition RF
3,0,0,0.977183,0.730170,0.835808,Test,RandomForestClassifier,Undersampling,Ignition RF
4,1,1,0.148571,0.495238,0.228571,Test,RandomForestClassifier,Undersampling,Ignition RF
5,2,2,0.225352,0.695652,0.340426,Test,RandomForestClassifier,Undersampling,Ignition RF
6,0,0,0.706923,0.657210,0.677852,Train,XGBClassifier,Undersampling,Ignition XGB
7,1,1,0.453563,0.459297,0.452355,Train,XGBClassifier,Undersampling,Ignition XGB
8,2,2,0.586785,0.613621,0.599207,Train,XGBClassifier,Undersampling,Ignition XGB
9,0,0,0.975806,0.717569,0.826997,Test,XGBClassifier,Undersampling,Ignition XGB


---

## 3. Export File

In [15]:
best_strategy_df.to_csv('../data/processed/ignition_best_strategy.csv',index=False)
print("All datasets saved successfully to ../data/processed/")

All datasets saved successfully to ../data/processed/
